# Agents as Tools

*Notebook 13*

Call specialists like functions and combine their outputs.

The orchestrator stays in control of the conversation.

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

import json
from agents import Agent, Runner, ToolCallItem

MODEL = "gpt-5-mini"


def function_tool_calls(result) -> list[tuple[str, dict]]:
    """Function-tool calls the run made, as (tool_name, arguments)."""
    calls = []
    for item in result.new_items:
        if isinstance(item, ToolCallItem):
            raw = item.raw_item
            if getattr(raw, "type", None) == "function_call":
                calls.append((raw.name, json.loads(raw.arguments)))
    return calls


print("✅ Ready!")

---

## 🎯 The Problem

Handoffs transfer active control to another agent.

Agents-as-tools return specialist output to the orchestrator, which can combine multiple results.

---

## 🔀 Part 1: Handoff vs Agent-as-Tool

Both patterns involve multiple agents.

The difference is who stays in control.

**Handoff**, transfer of control:
```
User → Triage → Specialist (takes over, handles response)
```

**Agent as Tool**, orchestrator stays in control:
```
User → Orchestrator → calls Specialist A → gets result back
→ calls Specialist B → gets result back
→ synthesizes both results → responds to user
```

<div style="text-align: left; display: inline-block;">

| | Handoff | Agent as Tool |
|---|---|---|
| Who responds | Active specialist | Original orchestrator |
| Original agent resumes automatically | No | Yes |
| Specialist input | Full history by default | Generated tool input |
| Best for | Transferring responsibility | Combining specialist outputs |

</div>

---

## 🛠️ Part 2: The `.as_tool()` Method

`.as_tool()` exposes any agent as a callable tool.

The specialist runs and returns its output, and the orchestrator continues from there.

To use it: build a specialist, then pass `specialist_agent.as_tool(...)` in another agent's `tools=[...]`.

Keep the specialist's instructions general enough for any input the orchestrator sends.

In [ ]:
# Two specialist agents
pros_agent = Agent(
    name="ProsAnalyst",
    instructions="You identify and explain the advantages and benefits of any topic. Be specific and concise.",
    model=MODEL
)

cons_agent = Agent(
    name="ConsAnalyst",
    instructions="You identify and explain the disadvantages and risks of any topic. Be specific and concise.",
    model=MODEL
)

# --------------------------------------------------------------
print("✅ Specialist agents defined")

#### Define the Orchestrator

In [ ]:
orchestrator_instructions = (
    "You provide balanced analysis on any topic.\n"
    "Always call both the pros_analyst and cons_analyst tools, then\n"
    "synthesize their outputs into a balanced summary with clear sections."
)

analysis_orchestrator = Agent(
    name="AnalysisOrchestrator",
    instructions=orchestrator_instructions,
    model=MODEL,
    tools=[
        pros_agent.as_tool(
            tool_name="pros_analyst",
            tool_description="Analyzes advantages and benefits of a topic."
        ),
        cons_agent.as_tool(
            tool_name="cons_analyst",
            tool_description="Analyzes disadvantages and risks of a topic."
        )
    ]
)

# `tool_name` gives the orchestrator a stable, function-like name to call.
# It does not have to match the specialist agent's display name.

# --------------------------------------------------------------
print("✅ Orchestrator ready")
print("   Tools: pros_analyst, cons_analyst")

#### Run the Orchestrator

In [ ]:
result = await Runner.run(analysis_orchestrator, input="Working from home vs working in an office")

print("🔀 ORCHESTRATOR RESULT")
print("=" * 60)
print(f"Handled by: {result.last_agent.name}")
print()
print(result.final_output)
print("=" * 60)

# Evidence: which specialist tools the orchestrator actually called
called = {name for name, _ in function_tool_calls(result)}
for tool in ["pros_analyst", "cons_analyst"]:
    print(f"{tool} called: {'yes' if tool in called else 'no'}")

### 💡 Key Takeaway

The markers above show whether both specialists were called in this run.

`result.last_agent.name` is the orchestrator here, not a specialist.

Control never transferred away.

---

## 🏗️ Part 3: Multi-Specialist Orchestration

The value of agents-as-tools is letting the model direct the specialist calls and combining distinct perspectives.

In [ ]:
# Three specialist agents for a product review system
features_agent = Agent(
    name="FeaturesAnalyst",
    instructions="You analyze product features. List and explain the key capabilities concisely.",
    model=MODEL
)

pricing_agent = Agent(
    name="PricingAnalyst",
    instructions="You analyze pricing and value. Comment on cost, tiers, and whether the pricing is fair.",
    model=MODEL
)

verdict_agent = Agent(
    name="VerdictWriter",
    instructions="You write final verdicts. Given a summary of features and pricing, write a clear buy/skip recommendation with reasoning.",
    model=MODEL
)

# --------------------------------------------------------------
print("✅ Specialist agents defined")

#### Define Review Instructions

In [ ]:
review_instructions = (
    "You produce comprehensive product reviews.\n"
    "For any product:\n"
    "1. Call features_analyst to analyze the features\n"
    "2. Call pricing_analyst to evaluate the pricing\n"
    "3. Call verdict_writer. In its input, include both analyses using these exact labels:\n"
    "   'Features analysis:' then the features_analyst output, and\n"
    "   'Pricing analysis:' then the pricing_analyst output. Then use its recommendation\n"
    "4. Present all three sections clearly to the user."
)

review_orchestrator = Agent(
    name="ReviewOrchestrator",
    instructions=review_instructions,
    model=MODEL,
    tools=[
        features_agent.as_tool(
            tool_name="features_analyst",
            tool_description="Analyzes product features and capabilities."
        ),
        pricing_agent.as_tool(
            tool_name="pricing_analyst",
            tool_description="Evaluates product pricing and value for money."
        ),
        verdict_agent.as_tool(
            tool_name="verdict_writer",
            tool_description="Writes a final buy/skip verdict given feature and pricing analysis."
        )
    ]
)

# --------------------------------------------------------------
print("✅ Review orchestrator ready")

#### Run the Review Pipeline

In [ ]:
product = "A wireless noise-cancelling headphone, 30-hour battery life, $249"

result = await Runner.run(review_orchestrator, input=product)

print("📋 PRODUCT REVIEW")
print("=" * 60)
print(f"Handled by: {result.last_agent.name}")
print()
print(result.final_output)
print("=" * 60)

# Evidence: which specialists ran, and whether verdict_writer received both analyses
calls = function_tool_calls(result)
called = {name for name, _ in calls}
for tool in ["features_analyst", "pricing_analyst", "verdict_writer"]:
    print(f"{tool} called: {'yes' if tool in called else 'no'}")

verdict_args = next((args for name, args in calls if name == "verdict_writer"), {})
verdict_input = verdict_args.get("input", "")
features_text = verdict_input.partition("Features analysis:")[2]
features_text, separator, pricing_text = features_text.partition("Pricing analysis:")
has_both = bool(separator and features_text.strip() and pricing_text.strip())
print(f"verdict_writer received both labeled analyses: {'yes' if has_both else 'no'}")

### 💡 Key Takeaway

Earlier tool results stay in the orchestrator's context for the run.

When calling `verdict_writer`, pass the relevant feature and pricing analysis in the tool input.

You can also skip a final specialist and synthesize the report in the orchestrator.

---

## 💪 Practice Exercises

### Exercise 1: Trip Planner

*Covers: agents as tools, orchestrating specialist agents*

Build an orchestrator that combines an activities specialist and a food specialist for a day trip.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Trip Planner Orchestrator
# --------------------------------------------------------------
# Objective: Combine two specialists into one trip planning response.

# TODO 1: Create an ActivitiesExpert agent with instructions to suggest
#          sightseeing, outdoor, and cultural activities for a city

# TODO 2: Create a FoodExpert agent with instructions to suggest
#          restaurants, local dishes, and food experiences for a city

# TODO 3: Create a TripPlannerOrchestrator that calls both as tools, passing the
#          city explicitly to each, and combines their output into a one-day itinerary

# TODO 4: Run with: "Plan a day trip to New Orleans"
#          Print result.last_agent.name and result.final_output, and confirm both
#          specialist tools fired with function_tool_calls(result)

# --- Write your code below this line ---

### Exercise 2: Candidate Fit Summary

*Covers: agents as tools, multi-specialist orchestration*

Build an orchestrator that reviews a resume against a role using two specialists.

It produces a fit summary for a human reviewer.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Candidate Fit Summary
# --------------------------------------------------------------
# Objective: Coordinate two reviewers into a fit summary for a human reviewer.

resume = """
Candidate: Alex Chen
Experience: 3 years Python development, 1 year team lead
Skills: Python, FastAPI, PostgreSQL, Docker, AWS basics
Education: BS Computer Science
Projects: Built internal reporting tool used by 50+ employees
"""

job_requirements = """
Role: Senior Python Developer
Required: 4+ years Python, REST API experience, database skills
Nice to have: Cloud experience, leadership experience
"""

# TODO 1: Create a SkillsReviewer agent that checks technical fit

# TODO 2: Create an ExperienceReviewer agent that checks seniority fit

# TODO 3: Create a CandidateFitOrchestrator that calls both reviewers, passing the
#          resume and job requirements explicitly to each, and produces a fit
#          summary for a human reviewer:
#            - evidence-backed strengths
#            - requirement gaps
#            - questions for a human reviewer
#          Leave the final decision to the human reviewer.

# TODO 4: Run with the resume and job requirements above. Confirm both reviewer
#          tools fired with function_tool_calls(result)

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Agents-as-tools return control to the orchestrator:**

- Handoffs transfer control to a specialist

- Agents-as-tools return the specialist's output to the orchestrator, which stays in control

- The orchestrator can synthesize several specialists' outputs into one answer

- `result.last_agent.name` remains the orchestrator in this workflow
<br>
<br>

**Nested agents get generated tool input, not the outer history:**

- A specialist called as a tool receives only the input the orchestrator passes

- Pass prior specialist outputs explicitly when a later specialist needs them
<br>
<br>

**Expose and verify agent tools:**

- `agent.as_tool(tool_name=..., tool_description=...)` exposes any agent as a callable tool

- `tool_description` guides tool choice but does not enforce it

- Inspect function-tool calls when tool use matters

---

## 📍 Next Step

**Notebook 14: Parallel Execution**  

Here, the model decides which specialists to call and when.

Notebook 14 puts your code in charge of the fan-out: run independent agents concurrently with `asyncio`.

When calls overlap, this can lower wall time before their results are combined.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-13-agents-as-tools)

---